In [ ]:
# [1] GPU Check & Install
import torch, gc, subprocess, sys
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    p = torch.cuda.get_device_properties(0)
    print(f'[+] GPU: {p.name} | VRAM: {p.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('No GPU! Enable GPU in Colab: Runtime → Change runtime type → T4 GPU')
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless','scikit-image'], check=False)
print('[+] Packages ready!')

In [ ]:
# [2] Create Directories
import os
for d in ['scripts/checkpoints','models',
          'competition_data/public/noisy',
          'competition_data/public/ground_truth']:
    os.makedirs(d, exist_ok=True)
print('[+] Directories created!')

In [ ]:
# [3] Write scripts/models.py
import os; os.makedirs('scripts', exist_ok=True)
with open('scripts/models.py','w') as f: f.write("import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nclass SimpleGate(nn.Module):\n    def forward(self, x):\n        x1, x2 = x.chunk(2, dim=1)\n        return x1 * x2\n\nclass NAFBlock(nn.Module):\n    def __init__(self, c, DW_Expand=2, FFN_Expand=2, drop_out_rate=0.0):\n        super().__init__()\n        dw = c * DW_Expand\n        self.conv1 = nn.Conv2d(c, dw, 1, bias=True)\n        self.conv2 = nn.Conv2d(dw, dw, 3, 1, 1, groups=dw, bias=True)\n        self.conv3 = nn.Conv2d(dw//2, c, 1, bias=True)\n        self.sca   = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw//2, dw//2, 1, bias=True))\n        self.sg    = SimpleGate()\n        ffn        = FFN_Expand * c\n        self.conv4 = nn.Conv2d(c, ffn, 1, bias=True)\n        self.conv5 = nn.Conv2d(ffn//2, c, 1, bias=True)\n        self.norm1 = nn.GroupNorm(1, c)\n        self.norm2 = nn.GroupNorm(1, c)\n        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()\n        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()\n        self.beta  = nn.Parameter(torch.zeros(1, c, 1, 1))\n        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))\n\n    def forward(self, inp):\n        x = self.norm1(inp)\n        x = self.conv1(x); x = self.conv2(x); x = self.sg(x)\n        x = x * self.sca(x); x = self.conv3(x); x = self.dropout1(x)\n        y = inp + x * self.beta\n        x = self.norm2(y); x = self.conv4(x); x = self.sg(x)\n        x = self.conv5(x); x = self.dropout2(x)\n        return y + x * self.gamma\n\nclass NAFNet(nn.Module):\n    def __init__(self, img_channel=3, width=64,\n                 enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=8):\n        super().__init__()\n        self.intro  = nn.Conv2d(img_channel, width, 3, 1, 1, bias=True)\n        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1, bias=True)\n        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()\n        self.ups, self.downs = nn.ModuleList(), nn.ModuleList()\n        chan = width\n        for num in enc_blks:\n            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))\n            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))\n            chan *= 2\n        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])\n        for num in dec_blks:\n            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))\n            chan //= 2\n            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))\n        self.padder_size = 2 ** len(enc_blks)\n\n    def forward(self, inp):\n        B, C, H, W = inp.shape\n        ph = (self.padder_size - H % self.padder_size) % self.padder_size\n        pw = (self.padder_size - W % self.padder_size) % self.padder_size\n        x  = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp\n        res = x; x = self.intro(x); encs = []\n        for enc, down in zip(self.encoders, self.downs):\n            x = enc(x); encs.append(x); x = down(x)\n        x = self.middle_blks(x)\n        for dec, up, sk in zip(self.decoders, self.ups, reversed(encs)):\n            x = up(x) + sk; x = dec(x)\n        x = self.ending(x) + res\n        return torch.clamp(x[:,:,:H,:W] if ph or pw else x, 0., 1.)\n\ndef build_model(width=64):\n    m = NAFNet(img_channel=3, width=width,\n               enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=8)\n    n = sum(p.numel() for p in m.parameters())\n    print(f'[Model] NAFNet-MASTER | Width={width} | Params={n/1e6:.2f}M')\n    return m\n")
print('[+] scripts/models.py written!')

In [ ]:
# [4] Write scripts/dataset.py
with open('scripts/dataset.py','w') as f: f.write('import os, glob, random\nimport numpy as np, cv2, torch\nfrom torch.utils.data import Dataset\n\nclass PatchDataset(Dataset):\n    """Training: random crops with augmentation."""\n    def __init__(self, pairs, patch_size=256):\n        self.pairs = pairs\n        self.patch_size = patch_size\n        print(f\'[Dataset] PatchDataset: {len(pairs)} pairs | patch={patch_size}\')\n\n    def __len__(self): return len(self.pairs)\n\n    def _load(self, idx):\n        np_, gp = self.pairs[idx]\n        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)\n        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)\n        return n, g\n\n    def __getitem__(self, idx):\n        n, g = self._load(idx)\n        h, w = n.shape[:2]; p = self.patch_size\n        y = random.randint(0, max(0, h-p))\n        x = random.randint(0, max(0, w-p))\n        n, g = n[y:y+p, x:x+p], g[y:y+p, x:x+p]\n        if random.random() > 0.5: n, g = np.fliplr(n).copy(), np.fliplr(g).copy()\n        if random.random() > 0.5: n, g = np.flipud(n).copy(), np.flipud(g).copy()\n        k = random.randint(0, 3)\n        if k: n, g = np.rot90(n, k).copy(), np.rot90(g, k).copy()\n        return (torch.from_numpy(np.ascontiguousarray(n)).permute(2,0,1).float()/255.,\n                torch.from_numpy(np.ascontiguousarray(g)).permute(2,0,1).float()/255.)\n\nclass FullImageDataset(Dataset):\n    """Validation: full-resolution images, no augmentation."""\n    def __init__(self, pairs):\n        self.pairs = pairs\n        print(f\'[Dataset] FullImageDataset: {len(pairs)} pairs (full resolution)\')\n\n    def __len__(self): return len(self.pairs)\n\n    def __getitem__(self, idx):\n        np_, gp = self.pairs[idx]\n        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)\n        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)\n        return (torch.from_numpy(n).permute(2,0,1).float()/255.,\n                torch.from_numpy(g).permute(2,0,1).float()/255.,\n                os.path.basename(gp))\n\ndef build_pairs(noisy_dir, gt_dir):\n    gt_files = sorted(glob.glob(os.path.join(gt_dir, \'*.png\')))\n    pairs = []\n    for gp in gt_files:\n        bid = os.path.splitext(os.path.basename(gp))[0]\n        for suffix in [f\'{bid}_noise.png\', f\'{bid}.png\', f\'noisy_{bid}.png\']:\n            np_ = os.path.join(noisy_dir, suffix)\n            if os.path.exists(np_):\n                pairs.append((np_, gp)); break\n    print(f\'[Dataset] Found {len(pairs)} valid pairs out of {len(gt_files)} GT images\')\n    return pairs\n')
print('[+] scripts/dataset.py written!')

In [ ]:
# [5] Load Dataset via Google Drive Mount
from google.colab import drive
import os, glob, shutil
drive.mount('/content/drive')

NOISY_DIR = 'competition_data/public/noisy'
GT_DIR    = 'competition_data/public/ground_truth'
os.makedirs(NOISY_DIR, exist_ok=True)
os.makedirs(GT_DIR, exist_ok=True)

# Auto-find noisy/ground_truth anywhere in Drive
fn = glob.glob('/content/drive/MyDrive/**/noisy', recursive=True)
fg = glob.glob('/content/drive/MyDrive/**/ground_truth', recursive=True)

if fn:
    files = glob.glob(f'{fn[0]}/*.png')
    for f in files: shutil.copy(f, NOISY_DIR)
    print(f'[+] {len(files)} noisy images copied')
if fg:
    files = glob.glob(f'{fg[0]}/*.png')
    for f in files: shutil.copy(f, GT_DIR)
    print(f'[+] {len(files)} GT images copied')

nn_ = len(glob.glob(f'{NOISY_DIR}/*.png'))
ng  = len(glob.glob(f'{GT_DIR}/*.png'))
print(f'[+] Dataset: {nn_} noisy | {ng} GT')
if ng == 0: raise RuntimeError('No GT images found!')
print('[+] Dataset ready!')

In [ ]:
# [6] Load v10 Champion Weights
import os, glob, shutil, torch
V10_PATH = 'models/best_denoiser_v10.pth'
os.makedirs('models', exist_ok=True)

# Search in Drive
dv10 = glob.glob('/content/drive/MyDrive/**/best_denoiser_v10.pth', recursive=True)
if dv10:
    shutil.copy(dv10[0], V10_PATH)
    print(f'[+] Loaded v10 from Drive: {dv10[0]}')
elif not os.path.exists(V10_PATH):
    from google.colab import files
    print('[*] Upload best_denoiser_v10.pth from your PC:')
    uploaded = files.upload()
    for fn in uploaded:
        if fn.endswith('.pth'): shutil.move(fn, V10_PATH)

ckpt = torch.load(V10_PATH, map_location='cpu')
print('='*70)
print(f'[+] v10 READY: Score={ckpt["composite_score"]:.4f} | '
      f'PSNR={ckpt["psnr"]:.4f} dB | SSIM={ckpt["ssim"]:.4f}')
print('='*70)

In [ ]:
# [7] ===== v12 MASTER Training Engine =====
import sys; sys.path.insert(0, 'scripts')
from models import build_model
from dataset import PatchDataset, FullImageDataset, build_pairs
import torch, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR
import time, gc, os, copy, math, numpy as np

gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')

# ─── HYPER-PARAMETERS ────────────────────────────────────────────────
WIDTH        = 64
PATCH_SIZE   = 256
BATCH_SIZE   = 4
ACCUM_STEPS  = 4        # Effective batch = 16
TOTAL_EPOCHS = 100
WARMUP_EP    = 5        # LR ramps from 1e-6 → PEAK_LR over 5 epochs
PEAK_LR      = 2e-5    # Higher than v11 (5e-6 was too gentle)
MIN_LR       = 1e-7
EMA_DECAY    = 0.9995
SWA_START_EP = 81       # SWA over final 20 epochs
SWA_LR       = 5e-7
PATIENCE     = 40       # Early stopping
SSIM_WEIGHT  = 0.15     # Loss = Charbonnier + 0.15*SSIM_loss
VAL_SPLIT    = 20       # Hold out 20 images for validation
NOISY_DIR    = 'competition_data/public/noisy'
GT_DIR       = 'competition_data/public/ground_truth'
# ─────────────────────────────────────────────────────────────────────

# Build pairs and split
all_pairs = build_pairs(NOISY_DIR, GT_DIR)
import random; random.seed(42); random.shuffle(all_pairs)
val_pairs   = all_pairs[:VAL_SPLIT]
train_pairs = all_pairs[VAL_SPLIT:]
print(f'[+] Train: {len(train_pairs)} | Val: {len(val_pairs)} images')

train_ds  = PatchDataset(train_pairs, patch_size=PATCH_SIZE)
val_ds    = FullImageDataset(val_pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

# Build model + warm-start
model = build_model(width=WIDTH).to(device)
ckpt  = torch.load('models/best_denoiser_v10.pth', map_location='cpu')
sd    = {k.replace('module.',''):v for k,v in ckpt['model_state_dict'].items() if k != 'n_averaged'}
model.load_state_dict(sd, strict=False)
print(f'[*] WARM-START from v10: Score={ckpt["composite_score"]:.4f} | PSNR={ckpt["psnr"]:.4f}')

# EMA
class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters(): p.requires_grad_(False)
        self.decay = decay
    def update(self, model):
        with torch.no_grad():
            for ep, mp in zip(self.ema.parameters(), model.parameters()):
                ep.data.mul_(self.decay).add_(mp.data, alpha=1-self.decay)

ema       = ModelEMA(model, decay=EMA_DECAY)
swa_model = AveragedModel(model)
swa_active = False

optimizer     = optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-5)
cosine_sched  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS-WARMUP_EP, eta_min=MIN_LR)
swa_scheduler = SWALR(optimizer, swa_lr=SWA_LR, anneal_epochs=5)
scaler        = torch.amp.GradScaler('cuda')

# ─── LOSSES ──────────────────────────────────────────────────────────
def charbonnier(pred, gt, eps=1e-4):
    d = pred.float() - gt.float()
    return torch.mean(torch.sqrt(d*d + eps))

def ssim_loss(pred, gt):
    """Differentiable SSIM loss (minimize 1-SSIM)."""
    p, g = pred.float(), gt.float()
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(p, 11, 1, 5)
    mu_g = F.avg_pool2d(g, 11, 1, 5)
    sg_pp = F.avg_pool2d(p*p,11,1,5) - mu_p**2
    sg_gg = F.avg_pool2d(g*g,11,1,5) - mu_g**2
    sg_pg = F.avg_pool2d(p*g,11,1,5) - mu_p*mu_g
    ssim_map = ((2*mu_p*mu_g+C1)*(2*sg_pg+C2)) / ((mu_p**2+mu_g**2+C1)*(sg_pp+sg_gg+C2)+1e-8)
    return 1.0 - ssim_map.mean()

def combined_loss(pred, gt):
    return charbonnier(pred, gt) + SSIM_WEIGHT * ssim_loss(pred, gt)

# ─── METRICS ─────────────────────────────────────────────────────────
BASE_PSNR, BASE_SSIM = 19.8257, 0.4644

def psnr_fn(p, g):
    mse = torch.mean((p.float()-g.float())**2).clamp(1e-10)
    return 10.*math.log10(1./mse.item())

def ssim_fn(p, g):
    p, g = p.float(), g.float()
    C1, C2 = 0.01**2, 0.03**2
    mu_p = F.avg_pool2d(p,11,1,5); mu_g = F.avg_pool2d(g,11,1,5)
    sg_pp = F.avg_pool2d(p*p,11,1,5)-mu_p**2
    sg_gg = F.avg_pool2d(g*g,11,1,5)-mu_g**2
    sg_pg = F.avg_pool2d(p*g,11,1,5)-mu_p*mu_g
    return torch.mean((2*mu_p*mu_g+C1)*(2*sg_pg+C2)/((mu_p**2+mu_g**2+C1)*(sg_pp+sg_gg+C2)+1e-8)).item()

def composite(psnr, ssim):
    return 0.6*min(max((psnr-BASE_PSNR)/15.,0.),1.) + 0.4*max(ssim-BASE_SSIM,0.)

def full_image_eval(net):
    """PROPER evaluation on full-resolution validation images."""
    net.eval()
    psnrs, ssims = [], []
    with torch.no_grad():
        for noisy, gt, name in val_loader:
            noisy, gt = noisy.to(device), gt.to(device)
            with torch.amp.autocast('cuda'):
                pred = net(noisy)
            psnrs.append(psnr_fn(pred, gt))
            ssims.append(ssim_fn(pred, gt))
    vp = sum(psnrs)/len(psnrs)
    vs = sum(ssims)/len(ssims)
    return vp, vs, composite(vp, vs)

# ─── CHECKPOINT SAVING ───────────────────────────────────────────────
CKPT_V12 = 'scripts/checkpoints/best_denoiser_v12.pth'
DRIVE_PATH = '/content/drive/MyDrive/best_denoiser_v12.pth'
os.makedirs('scripts/checkpoints', exist_ok=True)

def save_ckpt(sd, psnr, ssim, score, epoch, tag):
    obj = {'model_state_dict':sd,'width':WIDTH,'psnr':psnr,
           'ssim':ssim,'composite_score':score,'epoch':epoch,'type':tag}
    torch.save(obj, CKPT_V12)
    try: torch.save(obj, DRIVE_PATH); print(f'    [Drive] Saved to {DRIVE_PATH}')
    except Exception as e: print(f'    [Drive] Save failed: {e}')

# ─── TRAINING LOOP ───────────────────────────────────────────────────
best_score = ckpt['composite_score']   # Must beat v10
best_psnr  = ckpt['psnr']
best_ssim  = ckpt['ssim']
no_improve = 0

print('='*85)
print(f'v12 MASTER | Ep={TOTAL_EPOCHS} | PEAK_LR={PEAK_LR} | SSIM_W={SSIM_WEIGHT}')
print(f'Train={len(train_pairs)} | Val={len(val_pairs)} | Patience={PATIENCE}')
print(f'Must beat: Score={best_score:.4f} | PSNR={best_psnr:.4f} | SSIM={best_ssim:.4f}')
print('='*85)

for epoch in range(1, TOTAL_EPOCHS+1):
    # Warmup LR
    if epoch <= WARMUP_EP:
        warmup_lr = 1e-6 + (PEAK_LR - 1e-6) * (epoch / WARMUP_EP)
        for pg in optimizer.param_groups: pg['lr'] = warmup_lr

    if epoch == SWA_START_EP:
        print(f'\n[SWA] Activated at Epoch {epoch}!')
        swa_active = True

    # Training step
    model.train(); total_loss = 0.0; optimizer.zero_grad()
    t0 = time.time()

    for i, (noisy, gt) in enumerate(train_loader):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(noisy)
            loss = combined_loss(pred, gt) / ACCUM_STEPS
        if torch.isnan(loss) or torch.isinf(loss): optimizer.zero_grad(); continue
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS
        if (i+1) % ACCUM_STEPS == 0 or (i+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad()
            ema.update(model)

    if swa_active:
        swa_model.update_parameters(model)
        swa_scheduler.step()
    elif epoch > WARMUP_EP:
        cosine_sched.step()

    cur_lr   = optimizer.param_groups[0]['lr']
    avg_loss = total_loss / len(train_loader)
    elapsed  = time.time() - t0

    # ── PROPER FULL-IMAGE VALIDATION ──
    model.eval(); torch.cuda.empty_cache()
    vp_s, vs_s, c_s = full_image_eval(model)
    vp_e, vs_e, c_e = full_image_eval(ema.ema)

    candidates = [(c_s, vp_s, vs_s, model.state_dict(), '[STD]'),
                  (c_e, vp_e, vs_e, ema.ema.state_dict(), '[EMA]')]

    if swa_active and epoch % 5 == 0:
        torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
        p_sw, s_sw, c_sw = full_image_eval(swa_model)
        candidates.append((c_sw, p_sw, s_sw, {k:v for k,v in swa_model.state_dict().items() if 'n_averaged' not in k}, '[SWA]'))

    cur_comp, cur_p, cur_s, best_w, tag = max(candidates, key=lambda x: x[0])
    flag = ''
    if cur_comp > best_score and not np.isnan(cur_comp):
        best_score, best_psnr, best_ssim = cur_comp, cur_p, cur_s
        save_ckpt(best_w, cur_p, cur_s, cur_comp, epoch, tag)
        flag = f'  ★ NEW BEST! → Score={cur_comp:.4f}'
        no_improve = 0
    else:
        no_improve += 1

    print(f'Ep [{epoch:03d}/{TOTAL_EPOCHS}] Loss={avg_loss:.5f} | '
          f'Val PSNR={cur_p:.4f} dB | SSIM={cur_s:.5f} | '
          f'Score={cur_comp:.4f} {tag} | LR={cur_lr:.1e} | {elapsed:.0f}s{flag}')
    torch.cuda.empty_cache()

    # Early stopping
    if no_improve >= PATIENCE:
        print(f'\n[Early Stop] No improvement for {PATIENCE} epochs. Stopping.')
        break

# ── FINAL SWA FLUSH ──
if swa_active:
    print('\n[*] Final SWA BN update...')
    torch.optim.swa_utils.update_bn(train_loader, swa_model, device=device)
    p_f, s_f, c_f = full_image_eval(swa_model)
    print(f'    SWA Final: PSNR={p_f:.4f} | SSIM={s_f:.4f} | Score={c_f:.4f}')
    if c_f > best_score:
        best_score, best_psnr, best_ssim = c_f, p_f, s_f
        swa_sd = {k:v for k,v in swa_model.state_dict().items() if 'n_averaged' not in k}
        save_ckpt(swa_sd, p_f, s_f, c_f, TOTAL_EPOCHS, '[SWA-FINAL]')

print('='*85)
print('v12 MASTER Training Complete!')
print(f'  Best Score : {best_score:.4f}  (v10 baseline: 0.6757)')
print(f'  Best PSNR  : {best_psnr:.4f} dB')
print(f'  Best SSIM  : {best_ssim:.4f}')
if best_score > 0.6757:
    print(f'  [SUCCESS] Beat v10 by +{best_score-0.6757:.4f}!')
else:
    print('  [INFO] v10 remains champion. Use best_denoiser_v10.pth for submission.')
print('='*85)

In [ ]:
# [8] Verify, Save to Drive, and Download to Computer
import hashlib, os, shutil
from google.colab import files

CKPT = 'scripts/checkpoints/best_denoiser_v12.pth'
if not os.path.exists(CKPT):
    print('[!] v12 checkpoint not saved — v10 is still champion.')
else:
    sha = hashlib.sha256()
    with open(CKPT,'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''): sha.update(chunk)
    sz = os.path.getsize(CKPT)/1024**2
    print('='*70)
    print(f'File      : best_denoiser_v12.pth')
    print(f'Size      : {sz:.1f} MB')
    print(f'SHA-256   : {sha.hexdigest()}')
    try: print(f'Score     : {best_score:.4f} | PSNR: {best_psnr:.4f} dB | SSIM: {best_ssim:.4f}')
    except: pass
    print('='*70)

    # Save to Drive
    try:
        shutil.copy2(CKPT, '/content/drive/MyDrive/best_denoiser_v12.pth')
        print('[+] Saved to Google Drive!')
    except Exception as e:
        print(f'[!] Drive save failed: {e}')

    # Download to PC
    print('[*] Downloading to your computer...')
    files.download(CKPT)
    print('[+] Done!')